In [ ]:
import pandas as pd

year_one = pd.read_excel('data.xlsx', sheet_name='Year 2009-2010')
year_two = pd.read_excel('data.xlsx', sheet_name='Year 2010-2011')

print("Both sheets loaded successfully!")

all_data = pd.concat([year_one, year_two], ignore_index=True)

print("Sheets combined")
print("Total number of rows we have now:", len(all_data))

print("Here are the columns:")
print(all_data.columns)


Both sheets loaded successfully!
Sheets combined
Total number of rows we have now: 1067371
Here are the columns:
Index(['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'Price', 'Customer ID', 'Country'],
      dtype='object')


In [ ]:
print(all_data.head(3))

  Invoice StockCode                          Description  Quantity  \
0  489434     85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS        12   
1  489434    79323P                   PINK CHERRY LIGHTS        12   
2  489434    79323W                  WHITE CHERRY LIGHTS        12   

          InvoiceDate  Price  Customer ID         Country  
0 2009-12-01 07:45:00   6.95      13085.0  United Kingdom  
1 2009-12-01 07:45:00   6.75      13085.0  United Kingdom  
2 2009-12-01 07:45:00   6.75      13085.0  United Kingdom  


In [ ]:
clean = all_data.dropna(subset=['Customer ID']).copy()
print("Rows left for analysis: ", len(clean))
clean = clean.rename(columns={
    'Invoice': 'Invoice Number',
    'Price': 'Unit Price',
    'Customer ID': 'CustomerID'
})
print("The columns are changed", list(clean.columns))

Rows left for analysis:  824364
The columns are changed ['Invoice Number', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Unit Price', 'CustomerID', 'Country']


In [ ]:
clean = clean[(clean['Quantity'] > 0) & (clean['Unit Price'] > 0)]
print("Data Filtered Successfully")
print("Final rows after cleaning", len(clean))
print("Minimum quantity in our table", clean['Quantity'].min())
print("Minimum Unit Price in our table", clean['Unit Price'].min())

Data Filtered Successfully
Final rows after cleaning 805549
Minimum quantity in our table 1
Minimum Unit Price in our table 0.001


In [ ]:
clean['Line Total'] = clean['Quantity'] * clean['Unit Price']
clean['InvoiceDate'] = pd.to_datetime(clean['InvoiceDate'])
print(clean[['InvoiceDate','Invoice Number', 'Quantity','Unit Price', 'Line Total']].head(3))

          InvoiceDate Invoice Number  Quantity  Unit Price  Line Total
0 2009-12-01 07:45:00         489434        12        6.95        83.4
1 2009-12-01 07:45:00         489434        12        6.75        81.0
2 2009-12-01 07:45:00         489434        12        6.75        81.0


In [ ]:
last_date = clean['InvoiceDate'].max()
anchor_date = last_date + pd.Timedelta(days=1)
print("The absolute last purchase in the dataset was on", last_date)
print("Our analytical baseline (Anchor Date) is set to:", anchor_date.date())

The absolute last purchase in the dataset was on 2011-12-09 12:50:00
Our analytical baseline (Anchor Date) is set to: 2011-12-10


In [ ]:
customer_group = clean.groupby('CustomerID')
recency = customer_group['InvoiceDate'].max().apply(lambda x: (anchor_date - x).days)
frequency = customer_group['Invoice Number'].nunique()
monetary = customer_group['Line Total'].sum()
country = customer_group['Country'].first()
table = pd.DataFrame({
    'Recency': recency,
    'Frequency': frequency,
    'Monetary': monetary,
    'Country': country
}).reset_index()
print(f"Total Unique Customers found over the 24-month span: {len(table):,}")
print(table.head(5))

Total Unique Customers found over the 24-month span: 5,878
   CustomerID  Recency  Frequency  Monetary         Country
0     12346.0      326         12  77556.46  United Kingdom
1     12347.0        2          8   5633.32         Iceland
2     12348.0       75          5   2019.40         Finland
3     12349.0       19          4   4428.69           Italy
4     12350.0      310          1    334.40          Norway


In [ ]:
table['R_Score'] = pd.qcut(table['Recency'], q=5, labels=[5, 4, 3, 2, 1]).astype(int)

table['F_Score'] = pd.qcut(table['Frequency'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5]).astype(int)

table['M_Score'] = pd.qcut(table['Monetary'], q=5, labels=[1, 2, 3, 4, 5]).astype(int)

print(table[['CustomerID', 'Recency', 'R_Score', 'Frequency', 'F_Score', 'Monetary', 'M_Score']].head(5))

   CustomerID  Recency  R_Score  Frequency  F_Score  Monetary  M_Score
0     12346.0      326        2         12        5  77556.46        5
1     12347.0        2        5          8        4   5633.32        5
2     12348.0       75        3          5        4   2019.40        4
3     12349.0       19        5          4        3   4428.69        5
4     12350.0      310        2          1        1    334.40        2


In [ ]:
table['R_Score'] = table['R_Score'].astype(int)
table['F_Score'] = table['F_Score'].astype(int)
table['M_Score'] = table['M_Score'].astype(int)

# 2. Create our custom function containing our business logic rules
def assign_segment(row):
    r = row['R_Score']
    f = row['F_Score']
    m = row['M_Score']

    # Rule 1: Shopped recently, buys often, spends a lot -> Champions
    if r >= 4 and f >= 4 and m >= 4:
        return 'Champions'

    # Rule 2: Shopped very recently, but hasn't bought many times yet -> Recent New Buyers
    elif r >= 4 and f <= 2:
        return 'Recent New Buyers'

    # Rule 3: Hasn't shopped in a long time, but historically bought frequently and spent heavily -> Can't Lose Them
    elif r <= 2 and f >= 4 and m >= 4:
        return 'Can\'t Lose Them'

    # Rule 4: Hasn't shopped in a long time and rarely ever bought anything -> Lost / Hibernating
    elif r <= 2 and f <= 2:
        return 'Lost / Hibernating'

    # Rule 5: Any customer who doesn't fit the extreme categories above -> At Risk / Mid-Tier
    else:
        return 'At Risk / Mid-Tier'

#Tell Python to apply this function row-by-row across our entire table
table['Customer_Segment'] = table.apply(assign_segment, axis=1)

#Print out a summary table counting how many customers fall into each category
print(table['Customer_Segment'].value_counts())

#Save this final dataset as a CSV file
table.to_csv('cleaned_customer_rfm_matrix_2years.csv', index=False)

🎉 Segmentation Complete! Let's check the volume breakdown:
Customer_Segment
At Risk / Mid-Tier    2385
Lost / Hibernating    1523
Champions             1300
Recent New Buyers      443
Can't Lose Them        227
Name: count, dtype: int64

💾 Success! Saved file out as 'cleaned_customer_rfm_matrix_2years.csv'.
Open the folder icon on the left-side panel of Colab to download your clean dataset!


In [ ]:
import sqlite3

conn = sqlite3.connect('customer_rfm.db')
table.to_sql('customer_rfm', conn, if_exists='replace', index=False)

sql_query1 = """

SELECT
  Customer_Segment,
  COUNT(CustomerID) AS Total_Customers,
  ROUND(SUM(Monetary), 2) AS Total_Segment_Revenue
FROM customer_rfm
GROUP BY Customer_Segment
ORDER BY Total_Segment_Revenue DESC;

"""

sql_result1 = pd.read_sql_query(sql_query1, conn)
print(sql_result1)

     Customer_Segment  Total_Customers  Total_Segment_Revenue
0           Champions             1300            12128115.57
1  At Risk / Mid-Tier             2385             3534686.40
2     Can't Lose Them              227             1018866.68
3  Lost / Hibernating             1523              667121.92
4   Recent New Buyers              443              394638.61


In [ ]:
sql_query2 = """
SELECT
    CustomerID,
    Country,
    Monetary AS Lifetime_Spend,
    Customer_Segment
FROM customer_rfm
WHERE Customer_Segment = 'Champions'
  AND Country = 'United Kingdom'
ORDER BY Monetary DESC
LIMIT 5;
"""

sql_result2 = pd.read_sql_query(sql_query2, conn)
print("List generated successfully")
print(sql_result2)

List generated successfully
   CustomerID         Country  Lifetime_Spend Customer_Segment
0     18102.0  United Kingdom       608821.65        Champions
1     17450.0  United Kingdom       246973.09        Champions
2     13694.0  United Kingdom       196482.81        Champions
3     17511.0  United Kingdom       175603.55        Champions
4     16684.0  United Kingdom       147142.77        Champions
